# 03 — Silver: Typed Game + Pitch Tables

Typed extraction from the bronze VARIANT payload into two silver Delta tables:
`game_data` (one row per game) and `pitch_data` (one row per pitch event).

## Why this matters — Well-Architected Framework

| WAF pillar | What this notebook does, and why it's good |
|------------|---------------------------------------------|
| **Reliability** | MD5 surrogate keys (`game_sk`, `pitch_sk`) = deterministic dedup. `RELY` PK/FK = optimizer hints + Catalog Explorer ERD auto-draws. `CHECK` constraints = *enforced* physics rules so a bad upstream batch can't silently poison the table. |
| **Data Governance** | Every column has a descriptive `COMMENT` — this is what Genie uses for SQL generation accuracy. A 30-second investment per column pays back thousands of AI queries. |
| **Performance Efficiency** | Liquid clustering on `(season, official_date)` for games, `(season, official_date, game_pk)` for pitches matches the predicate shape Lakeview and Genie emit — same table, dramatically less data scanned. |
| **Operational Excellence** | `INSERT OVERWRITE` = full refresh every run = idempotent. No state, no late-arriving-data choreography. If you need incremental later, switch to `MERGE` via DLT — but for this demo, simpler is better. |


In [1]:
from databricks.connect import DatabricksSession
from dotenv import load_dotenv
import os

load_dotenv()

spark = DatabricksSession.builder.serverless().getOrCreate()

UC_CATALOG    = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA     = os.getenv("UC_SCHEMA",  "mlb_gumbo_waf")
BRONZE_SCHEMA = f"{UC_SCHEMA}_bronze"
SILVER_SCHEMA = f"{UC_SCHEMA}_silver"

BRONZE = f"{UC_CATALOG}.{BRONZE_SCHEMA}.raw_gumbo"
print(f"Bronze: {BRONZE}")
print(f"Silver: {UC_CATALOG}.{SILVER_SCHEMA}")


Bronze: alexander_booth.mlb_gumbo_waf_bronze.raw_gumbo
Silver: alexander_booth.mlb_gumbo_waf_silver


In [2]:
def add_pk(table, name, cols):
    cat, sch, tbl = table.split(".")
    existing = spark.sql(f"""
        SELECT constraint_name FROM {cat}.information_schema.table_constraints
        WHERE table_schema = '{sch}' AND table_name = '{tbl}'
          AND constraint_type = 'PRIMARY KEY' AND constraint_name = '{name}'
    """).collect()
    if existing:
        print(f"  PK '{name}' already exists on {table}")
        return
    for c in cols:
        try:
            spark.sql(f"ALTER TABLE {table} ALTER COLUMN {c} SET NOT NULL")
        except Exception as e:
            if "ALREADY_NOT_NULLABLE" not in str(e):
                raise
    spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} PRIMARY KEY ({', '.join(cols)}) RELY")
    print(f"  Added PK '{name}' on {table}")


def add_fk(table, name, cols, ref_table, ref_cols):
    cat, sch, tbl = table.split(".")
    existing = spark.sql(f"""
        SELECT constraint_name FROM {cat}.information_schema.table_constraints
        WHERE table_schema = '{sch}' AND table_name = '{tbl}'
          AND constraint_type = 'FOREIGN KEY' AND constraint_name = '{name}'
    """).collect()
    if existing:
        print(f"  FK '{name}' already exists on {table}")
        return
    spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} FOREIGN KEY ({', '.join(cols)}) REFERENCES {ref_table} ({', '.join(ref_cols)})")
    print(f"  Added FK '{name}' on {table}")


def drop_check_if_exists(table, name):
    try:
        spark.sql(f"ALTER TABLE {table} DROP CONSTRAINT {name}")
    except Exception:
        pass


def add_check(table, name, predicate):
    # Drop-then-add is idempotent and lets us re-tune the predicate on re-runs.
    drop_check_if_exists(table, name)
    try:
        spark.sql(f"ALTER TABLE {table} ADD CONSTRAINT {name} CHECK ({predicate})")
        print(f"  Added CHECK '{name}' on {table}")
    except Exception as e:
        msg = str(e)
        if "VIOLATE" in msg.upper() or "violat" in msg:
            print(f"  CHECK '{name}' could not be added — existing data violates predicate:")
            print(f"    {msg[:200]}")
        else:
            raise

print("Helpers loaded.")


Helpers loaded.


## silver.game_data

One row per MLB game, typed out of the VARIANT payload. `game_sk = MD5(game_pk)`
is our stable surrogate key — it gives us a one-column PK even if the source
ever changes its integer ID shape.

In [3]:
game_table = f"{UC_CATALOG}.{SILVER_SCHEMA}.game_data"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {game_table} (
        game_sk               STRING   NOT NULL COMMENT 'MD5(game_pk) — stable surrogate key',
        game_pk               INT               COMMENT 'MLB Stats API primary key for the game',
        season                INT               COMMENT 'Season year (e.g. 2025)',
        official_date         DATE              COMMENT 'Official date of the game after any reschedule',
        game_type             STRING            COMMENT 'R=regular, D=division series, F=wildcard, L=LCS, W=World Series',
        home_team_id          INT               COMMENT 'MLB team ID of the home team',
        home_team_name        STRING            COMMENT 'Home team full name',
        away_team_id          INT               COMMENT 'MLB team ID of the away team',
        away_team_name        STRING            COMMENT 'Away team full name',
        venue_id              INT               COMMENT 'Venue ID',
        venue_name            STRING            COMMENT 'Venue name',
        weather_condition     STRING            COMMENT 'Reported weather (Clear, Cloudy, ...)',
        weather_temp_f        INT               COMMENT 'Reported temperature in Fahrenheit',
        weather_wind          STRING            COMMENT 'Reported wind, free-text (e.g. "5 mph, L To R")',
        attendance            INT               COMMENT 'Announced attendance',
        first_pitch           TIMESTAMP         COMMENT 'First pitch timestamp',
        status_detailed       STRING            COMMENT 'Detailed game status',
        flags_no_hitter       BOOLEAN           COMMENT 'True if the game ended as a no-hitter',
        flags_perfect_game    BOOLEAN           COMMENT 'True if the game ended as a perfect game',
        source_file           STRING            COMMENT 'Path to the bronze source file',
        file_batch_time       TIMESTAMP         COMMENT 'Bronze ingest-batch timestamp'
    )
    CLUSTER BY (season, official_date)
    COMMENT 'Silver — one row per MLB game, typed + deduped from bronze.raw_gumbo.'
""")
print(f"Table ready: {game_table}")


Table ready: alexander_booth.mlb_gumbo_waf_silver.game_data


In [4]:
spark.sql(f"""
    INSERT OVERWRITE {game_table}
    SELECT
        MD5(CAST(data:gamePk::int AS STRING))                           AS game_sk,
        data:gamePk::int                                                AS game_pk,
        data:gameData:game:season::int                                  AS season,
        data:gameData:datetime:officialDate::date                       AS official_date,
        data:gameData:game:type::string                                 AS game_type,
        data:gameData:teams:home:id::int                                AS home_team_id,
        data:gameData:teams:home:name::string                           AS home_team_name,
        data:gameData:teams:away:id::int                                AS away_team_id,
        data:gameData:teams:away:name::string                           AS away_team_name,
        data:gameData:venue:id::int                                     AS venue_id,
        data:gameData:venue:name::string                                AS venue_name,
        data:gameData:weather:condition::string                         AS weather_condition,
        data:gameData:weather:temp::int                                 AS weather_temp_f,
        data:gameData:weather:wind::string                              AS weather_wind,
        data:gameData:gameInfo:attendance::int                          AS attendance,
        data:gameData:gameInfo:firstPitch::timestamp                    AS first_pitch,
        data:gameData:status:detailedState::string                      AS status_detailed,
        COALESCE(data:gameData:flags:noHitter::boolean, FALSE)          AS flags_no_hitter,
        COALESCE(data:gameData:flags:perfectGame::boolean, FALSE)       AS flags_perfect_game,
        source_file,
        file_batch_time
    FROM {BRONZE}
    WHERE data:gamePk IS NOT NULL
    QUALIFY ROW_NUMBER() OVER (PARTITION BY data:gamePk::int ORDER BY file_batch_time DESC) = 1
""")
print(f"Rows: {spark.table(game_table).count():,}")


Rows: 2,477


## silver.pitch_data

One row per pitch event extracted from the `liveData.plays.allPlays[*].playEvents[*]`
arrays. We explode the JSON arrays with `variant_explode`.

In [5]:
pitch_table = f"{UC_CATALOG}.{SILVER_SCHEMA}.pitch_data"

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {pitch_table} (
        pitch_sk                STRING   NOT NULL COMMENT 'MD5(game_pk, at_bat_index, pitch_index)',
        game_sk                 STRING            COMMENT 'FK to silver.game_data.game_sk',
        game_pk                 INT               COMMENT 'MLB game primary key',
        season                  INT               COMMENT 'Season year',
        official_date           DATE              COMMENT 'Official game date',
        at_bat_index            INT               COMMENT '0-based index of the at-bat within the game',
        pitch_index             INT               COMMENT '0-based index of the event within the at-bat (pitches + actions)',
        inning                  INT               COMMENT 'Inning number',
        half_inning             STRING            COMMENT 'top or bottom',
        is_pitch                BOOLEAN           COMMENT 'True for actual pitches, false for pickoffs / mound visits',
        pitch_type_code         STRING            COMMENT 'Pitch-type code (FF, SL, CH, ...)',
        pitch_type_description  STRING            COMMENT 'Pitch-type human name (Four-Seam Fastball, Slider, ...)',
        call_code               STRING            COMMENT 'Call code (B=ball, S=strike, X=in play, ...)',
        call_description        STRING            COMMENT 'Call description (Ball, Called Strike, In play, run(s), ...)',
        is_strike               BOOLEAN           COMMENT 'True if the pitch resulted in a strike',
        is_in_play              BOOLEAN           COMMENT 'True if the ball was put in play',
        start_speed_mph         DOUBLE            COMMENT 'Pitch speed at release in mph',
        end_speed_mph           DOUBLE            COMMENT 'Pitch speed crossing the plate in mph',
        spin_rate_rpm           INT               COMMENT 'Spin rate in RPM',
        plate_x                 DOUBLE            COMMENT 'Horizontal plate location in feet (catcher POV)',
        plate_z                 DOUBLE            COMMENT 'Vertical plate location in feet from ground',
        sz_top                  DOUBLE            COMMENT 'Top of strike zone in feet',
        sz_bot                  DOUBLE            COMMENT 'Bottom of strike zone in feet',
        break_vertical_induced  DOUBLE            COMMENT 'Induced vertical break in inches',
        break_horizontal        DOUBLE            COMMENT 'Horizontal break in inches',
        extension_ft            DOUBLE            COMMENT 'Pitcher extension at release in feet',
        pitcher_id              INT               COMMENT 'MLB player ID of the pitcher',
        pitcher_name            STRING            COMMENT 'Pitcher full name',
        batter_id               INT               COMMENT 'MLB player ID of the batter',
        batter_name             STRING            COMMENT 'Batter full name',
        batter_side             STRING            COMMENT 'Batter stance: L or R',
        pitcher_hand            STRING            COMMENT 'Pitcher throwing hand: L or R'
    )
    CLUSTER BY (season, official_date, game_pk)
    COMMENT 'Silver — one row per pitch event, extracted from bronze GUMBO payload.'
""")
print(f"Table ready: {pitch_table}")

# Drop any pre-existing CHECK constraints so the INSERT OVERWRITE below is never
# blocked by a rule tuned against an older/smaller dataset. They get re-added
# (with whatever predicate we tune them to now) a few cells down.
for c in ["chk_pitch_speed_sane", "chk_pitch_spin_sane"]:
    drop_check_if_exists(pitch_table, c)
drop_check_if_exists(game_table, "chk_game_weather_sane")


Table ready: alexander_booth.mlb_gumbo_waf_silver.pitch_data


In [6]:
# Single flat SELECT with two LATERAL variant_explodes — same pattern as the
# original GUMBO E2E demo. No CTE, no QUALIFY: the optimizer fuses the two
# explodes into one pass.
spark.sql(f"""
    INSERT OVERWRITE {pitch_table}
    SELECT
        MD5(CONCAT_WS('|',
            CAST(data:gamePk::int AS STRING),
            CAST(allPlays.value:about:atBatIndex::int AS STRING),
            CAST(playEvents.value:index::int AS STRING)
        ))                                                        AS pitch_sk,
        MD5(CAST(data:gamePk::int AS STRING))                     AS game_sk,
        data:gamePk::int                                          AS game_pk,
        data:gameData:game:season::int                            AS season,
        data:gameData:datetime:officialDate::date                 AS official_date,
        allPlays.value:about:atBatIndex::int                      AS at_bat_index,
        playEvents.value:index::int                               AS pitch_index,
        allPlays.value:about:inning::int                          AS inning,
        allPlays.value:about:halfInning::string                   AS half_inning,
        playEvents.value:isPitch::boolean                         AS is_pitch,
        playEvents.value:details:type:code::string                AS pitch_type_code,
        playEvents.value:details:type:description::string         AS pitch_type_description,
        playEvents.value:details:call:code::string                AS call_code,
        playEvents.value:details:call:description::string         AS call_description,
        playEvents.value:details:isStrike::boolean                AS is_strike,
        playEvents.value:details:isInPlay::boolean                AS is_in_play,
        playEvents.value:pitchData:startSpeed::double             AS start_speed_mph,
        playEvents.value:pitchData:endSpeed::double               AS end_speed_mph,
        playEvents.value:pitchData:breaks:spinRate::int           AS spin_rate_rpm,
        playEvents.value:pitchData:coordinates:pX::double         AS plate_x,
        playEvents.value:pitchData:coordinates:pZ::double         AS plate_z,
        playEvents.value:pitchData:strikeZoneTop::double          AS sz_top,
        playEvents.value:pitchData:strikeZoneBottom::double       AS sz_bot,
        playEvents.value:pitchData:breaks:breakVerticalInduced::double AS break_vertical_induced,
        playEvents.value:pitchData:breaks:breakHorizontal::double AS break_horizontal,
        playEvents.value:pitchData:extension::double              AS extension_ft,
        allPlays.value:matchup:pitcher:id::int                    AS pitcher_id,
        allPlays.value:matchup:pitcher:fullName::string           AS pitcher_name,
        allPlays.value:matchup:batter:id::int                     AS batter_id,
        allPlays.value:matchup:batter:fullName::string            AS batter_name,
        allPlays.value:matchup:batSide:code::string               AS batter_side,
        allPlays.value:matchup:pitchHand:code::string             AS pitcher_hand
    FROM {BRONZE},
         LATERAL variant_explode(data:liveData:plays:allPlays)    AS allPlays,
         LATERAL variant_explode(allPlays.value:playEvents)       AS playEvents
    WHERE playEvents.value:isPitch::boolean = TRUE
""")
print(f"Rows: {spark.table(pitch_table).count():,}")


Rows: 724,180


## Constraints — PK, FK, CHECK

PK/FK are `RELY`-only (informational) — they feed the Catalog Explorer ERD and
let the optimizer drop redundant joins. CHECK constraints are real guardrails —
they'll block any write that violates them, which is exactly what we want for
physical-plausibility rules (a pitch can't go faster than 110 mph, etc.).

In [7]:
print("Primary keys...")
add_pk(game_table,  "game_data_pk",  ["game_sk"])
add_pk(pitch_table, "pitch_data_pk", ["pitch_sk"])

print("\nForeign keys (RELY)...")
add_fk(pitch_table, "pitch_data_game_fk", ["game_sk"], game_table, ["game_sk"])

print("\nCHECK constraints (enforced)...")
# Range covers real edge cases: a 21 mph eephus pitch is legitimate; anything
# above ~110 mph is a data bug. The ranges here are "flag the impossible",
# not "flag the rare."
add_check(game_table,  "chk_game_weather_sane", "weather_temp_f IS NULL OR (weather_temp_f BETWEEN -20 AND 130)")
add_check(pitch_table, "chk_pitch_speed_sane",  "start_speed_mph IS NULL OR (start_speed_mph BETWEEN 15 AND 120)")
add_check(pitch_table, "chk_pitch_spin_sane",   "spin_rate_rpm   IS NULL OR (spin_rate_rpm BETWEEN 0 AND 5000)")


Primary keys...


  Added PK 'game_data_pk' on alexander_booth.mlb_gumbo_waf_silver.game_data


  Added PK 'pitch_data_pk' on alexander_booth.mlb_gumbo_waf_silver.pitch_data

Foreign keys (RELY)...


  Added FK 'pitch_data_game_fk' on alexander_booth.mlb_gumbo_waf_silver.pitch_data

CHECK constraints (enforced)...


  Added CHECK 'chk_game_weather_sane' on alexander_booth.mlb_gumbo_waf_silver.game_data


  Added CHECK 'chk_pitch_speed_sane' on alexander_booth.mlb_gumbo_waf_silver.pitch_data


  Added CHECK 'chk_pitch_spin_sane' on alexander_booth.mlb_gumbo_waf_silver.pitch_data


## Verify

In [8]:
print("=" * 60)
print("SILVER LAYER SUMMARY")
print("=" * 60)

for table in [game_table, pitch_table]:
    n = spark.table(table).count()
    print(f"  {table}: {n:,} rows")

print("\nSanity: pitch count per game (top 5)")
spark.sql(f"""
    SELECT g.official_date, g.home_team_name, g.away_team_name,
           COUNT(*) AS pitch_count
    FROM {pitch_table} p JOIN {game_table} g USING (game_sk)
    GROUP BY ALL
    ORDER BY pitch_count DESC
    LIMIT 5
""").show(truncate=False)


SILVER LAYER SUMMARY


  alexander_booth.mlb_gumbo_waf_silver.game_data: 2,477 rows


  alexander_booth.mlb_gumbo_waf_silver.pitch_data: 724,180 rows

Sanity: pitch count per game (top 5)


+-------------+---------------------+-------------------+-----------+
|official_date|home_team_name       |away_team_name     |pitch_count|
+-------------+---------------------+-------------------+-----------+
|2025-04-06   |Boston Red Sox       |St. Louis Cardinals|742        |
|2025-07-02   |Washington Nationals |Detroit Tigers     |696        |
|2025-05-29   |Philadelphia Phillies|Atlanta Braves     |665        |
|2025-05-14   |Baltimore Orioles    |Minnesota Twins    |646        |
|2025-07-11   |Chicago White Sox    |Cleveland Guardians|644        |
+-------------+---------------------+-------------------+-----------+



> ## WAF takeaway — what to say on this slide
>
> - **Three kinds of constraints are doing three different jobs here:**
>   - `RELY PK` → optimizer + ERD (no runtime cost)
>   - `RELY FK` → optimizer + ERD (no runtime cost)
>   - `CHECK`   → **enforced** on every write — these are real guardrails
> - The next notebook (`06_reliability_dq`) goes beyond `CHECK` for the
>   rules that are too complex for a single predicate — cross-table
>   referential integrity, freshness, row-count anomalies. Layered defence.
